In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from datetime import datetime
from pathlib import Path

current_dir = Path.cwd().resolve()
for parent in [current_dir, *current_dir.parents]:
    if (parent / "src").exists():
        project_root = str(parent)
        break
else:
    project_root = str(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.utils.setup import setup_device, setup_directories
from src.utils.dataloader import BreastMNISTLoader, BCDRLoader, INbreastLoader, BUSUCLMLoader
from src.utils.training import training_loop, test_model
from src.models.main.models import QuantumModel, ExtentedQuantumModel

print("✓ Imports successful")

In [ ]:
CONFIG = {

'model_type': 'basic',
'num_classes': 2,
'input_channels': 1,


'dataset': 'breastmnist',
'batch_size': 16,
'image_size': 28,


'epochs': 20,
'learning_rate': 1e-3,


# 'basic' encodes one pixel per qubit, so n_qubits must equal patch_size ** 2.
'n_qubits': 4,
'n_layers': 2,
'patch_size': 2,
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
device = setup_device()
dirs = setup_directories()


dataset_mapping = {
'breastmnist': 'BreastMNIST',
'bcdr': 'BCDR',
'inbreast': 'INbreast',
'busuclm': 'BUSUCLM'
}
dataset_name = dataset_mapping.get(CONFIG['dataset'], 'BreastMNIST')
model_type = CONFIG['model_type']
save_dir = os.path.join(
dirs.get("project_dir", "."),
"experiments", "models", dataset_name, "quantum", model_type
)
os.makedirs(save_dir, exist_ok=True)

print(f"Device: {device}")
print(f"Save directory: {save_dir}")

In [ ]:
if CONFIG['dataset'] == 'bcdr':
    loader = BCDRLoader(
    batch_size=CONFIG['batch_size'],
    image_size=CONFIG['image_size'],
    root_dir='src/data/BCDR/png'
    )
    print(f"✓ BCDR dataset loaded")
elif CONFIG['dataset'] == 'inbreast':
    loader = INbreastLoader(
    batch_size=CONFIG['batch_size'],
    image_size=CONFIG['image_size'],
    root_dir='src/data/INbreast/png'
    )
    print(f"✓ INbreast dataset loaded")
elif CONFIG['dataset'] == 'busuclm':
    loader = BUSUCLMLoader(
    batch_size=CONFIG['batch_size'],
    image_size=CONFIG['image_size'],
    root_dir='src/data/BUS-UCLM/png'
    )
    print(f"✓ BUSUCLM dataset loaded")
else:
    loader = BreastMNISTLoader(
    batch_size=CONFIG['batch_size'],
    image_size=CONFIG['image_size'],
    )
    print(f"✓ BreastMNIST dataset loaded")

train_loader, test_loader, val_loader = loader.get_dataloaders()
dataloader = {"train": train_loader, "val": val_loader, "test": test_loader}

print(f"  Training batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

In [ ]:
sample_images, _ = next(iter(train_loader))
input_h, input_w = sample_images.shape[-2], sample_images.shape[-1]
output_size = (
max(1, input_h // CONFIG['patch_size']),
max(1, input_w // CONFIG['patch_size'])
)
rand_params = np.random.uniform(
high=2 * np.pi,
size=(CONFIG['n_layers'], CONFIG['n_qubits'])
)

print(f"Input size: {input_h}x{input_w} | Quantum output_size: {output_size}")

In [ ]:
if CONFIG['model_type'] == 'trainable':
    model = ExtentedQuantumModel(
    n_qubits=CONFIG['n_qubits'],
    n_layers=CONFIG['n_layers'],
    num_classes=CONFIG['num_classes'],
    patch_size=CONFIG['patch_size'],
    mode="2d",
    stride=None,
    use_parallel=True,
    ).to(device)
    model_name = 'extended_quantum'
else:
    if CONFIG['n_qubits'] != CONFIG['patch_size'] ** 2:
        raise ValueError(
        f"'basic' feeds one patch pixel per qubit, so n_qubits must equal patch_size ** 2 "
        f"(got n_qubits={CONFIG['n_qubits']}, patch_size={CONFIG['patch_size']} -> "
        f"{CONFIG['patch_size'] ** 2} pixels per patch)."
        )
    model = QuantumModel(
    n_qubits=CONFIG['n_qubits'],
    n_layers=CONFIG['n_layers'],
    rand_params=rand_params,
    output_size=output_size,
    patch_size=CONFIG['patch_size'],
    num_classes=CONFIG['num_classes']
    ).to(device)
    # QuantumModel builds its FC layer lazily, on the first forward pass.
    with torch.no_grad():
        _ = model(torch.randn(1, CONFIG['input_channels'], input_h, input_w).to(device))
    model_name = 'quantum'

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ Model created: {model.__class__.__name__}")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])

print(f"✓ Training setup complete")
print(f"  Loss: CrossEntropyLoss")
print(f"  Optimizer: Adam (lr={CONFIG['learning_rate']})")
print(f"  Epochs: {CONFIG['epochs']}")

In [ ]:
print(f"\n{'='*80}")
print(f"Starting Training - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*80}\n")

history = training_loop(
model=model,
train_loader=train_loader,
val_loader=val_loader,
criterion=criterion,
optimizer=optimizer,
device=device,
num_epochs=CONFIG['epochs'],
save_dir=save_dir,
model_name=model_name,
extract_features=False,
save_best_only=True,
save_all_epochs=True
)

print(f"\n{'='*80}")
print(f"Training Complete - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*80}")

In [ ]:
# model_path = "/home/yrsobrinho/HybridFeatureFusionForBreastCancer/experiments/models/BreastMNIST/quantum/basic/checkpoints/quantum_epoch4.pth"
model_path = os.path.join(save_dir, f"best_{model_name}.pth")

if os.path.isfile(model_path):
    state = torch.load(model_path, map_location=device, weights_only=True)
    model.load_state_dict(state)
    print(f"✓ Loaded best model from {model_path}")
else:
    print("⚠ Warning: No best model found, using final epoch weights")

model.eval()

In [ ]:
print(f"\n{'='*80}")
print("Testing Best Model")
print(f"{'='*80}\n")

test_result = test_model(
model=model,
dataloader=test_loader,
criterion=criterion,
device=device,
hyperparams=CONFIG,
output_file=os.path.join(save_dir, f"test_log_{model_name}.txt"),
extract_features=False,
compute_detailed_metrics=True
)


test_loss, test_acc, test_prec, test_rec, test_f1, test_auc = test_result


test_metrics = {
'loss': float(test_loss),
'accuracy': float(test_acc),
'precision': float(test_prec),
'recall': float(test_rec),
'f1': float(test_f1),
'auc': float(test_auc)
}

print("\nTest Results:")
print(f"  Loss: {test_metrics['loss']:.4f}")
print(f"  Accuracy: {test_metrics['accuracy']:.4f}")
print(f"  Precision: {test_metrics['precision']:.4f}")
print(f"  Recall: {test_metrics['recall']:.4f}")
print(f"  F1 Score: {test_metrics['f1']:.4f}")
print(f"  AUC: {test_metrics['auc']:.4f}")

In [ ]:
results_file = os.path.join(save_dir, f'test_results_{model_name}.txt')

with open(results_file, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write(f"TEST RESULTS - Quantum Model WITHOUT Encoder\n")
    f.write("="*80 + "\n\n")

    f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")

    f.write("MODEL CONFIGURATION\n")
    f.write("-"*80 + "\n")
    f.write(f"Model Type: {model.__class__.__name__}\n")
    f.write(f"Model Variant: {CONFIG['model_type']}\n")
    f.write(f"Dataset: {CONFIG['dataset'].upper()}\n")
    f.write(f"Encoder: None (end-to-end training)\n")
    f.write(f"Total Parameters: {total_params:,}\n")
    f.write(f"Trainable Parameters: {trainable_params:,}\n\n")

    f.write("QUANTUM CIRCUIT\n")
    f.write("-"*80 + "\n")
    f.write(f"Qubits: {CONFIG['n_qubits']}\n")
    f.write(f"Layers: {CONFIG['n_layers']}\n")
    f.write(f"Patch Size: {CONFIG['patch_size']}\n")
    f.write(f"Quantum Output Size: {output_size}\n\n")

    f.write("TRAINING HYPERPARAMETERS\n")
    f.write("-"*80 + "\n")
    f.write(f"Epochs: {CONFIG['epochs']}\n")
    f.write(f"Batch Size: {CONFIG['batch_size']}\n")
    f.write(f"Learning Rate: {CONFIG['learning_rate']}\n")
    f.write(f"Image Size: {CONFIG['image_size']}\n")
    f.write(f"Input Channels: {CONFIG['input_channels']}\n")
    f.write(f"Number of Classes: {CONFIG['num_classes']}\n\n")

    f.write("BEST VALIDATION METRICS\n")
    f.write("-"*80 + "\n")
    f.write(f"Best Validation Accuracy: {max(history['val_acc']):.4f}\n")
    f.write(f"Best Validation F1 Score: {max(history['val_f1']):.4f}\n")
    f.write(f"Best Validation AUC: {max(history['val_auc']):.4f}\n")
    f.write(f"Best Validation Precision: {max(history['val_precision']):.4f}\n")
    f.write(f"Best Validation Recall: {max(history['val_recall']):.4f}\n\n")

    f.write("FINAL TEST METRICS\n")
    f.write("-"*80 + "\n")
    f.write(f"Test Loss: {test_metrics['loss']:.4f}\n")
    f.write(f"Test Accuracy: {test_metrics['accuracy']:.4f}\n")
    f.write(f"Test Precision: {test_metrics['precision']:.4f}\n")
    f.write(f"Test Recall: {test_metrics['recall']:.4f}\n")
    f.write(f"Test F1 Score: {test_metrics['f1']:.4f}\n")
    f.write(f"Test AUC: {test_metrics['auc']:.4f}\n\n")

    f.write("MODEL SAVE LOCATION\n")
    f.write("-"*80 + "\n")
    f.write(f"Directory: {save_dir}\n")
    f.write(f"Best Model: best_{model_name}.pth\n")
    f.write(f"Epoch Checkpoints: checkpoints/{model_name}_epoch{{N}}.pth\n\n")

    f.write("="*80 + "\n")
    f.write("END OF REPORT\n")
    f.write("="*80 + "\n")

print(f"✓ Test results saved to: {results_file}")

## Multi-run evaluation

Trains the model above **N_RUNS** times from scratch (seeds `0..N_RUNS-1`) and reports **mean ± std** plus a **95% confidence interval** (Student's t) for every test metric, together with per-run and aggregated confusion matrices.

All artefacts are written under `reports/quantum/` (see `reports/README.md`). Adjust `N_RUNS` below if this is too slow — quantum runs in particular are expensive.

In [23]:
# ==========================================================================
# MULTI-RUN EVALUATION  (N_RUNS independent runs, seeds 0..N_RUNS-1)
# Retrains the model from scratch each run, then reports mean +/- std and a
# 95% CI (Student's t) for every test metric, plus per-run and aggregated
# confusion matrices. Everything is written under `reports/quantum/`.
# Run the setup cells above first (imports, data loaders, config, model class).
# ==========================================================================
import os
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.stats import t as t_dist
from sklearn.metrics import confusion_matrix

N_RUNS = 5
CONFIDENCE_LEVEL = 0.95
_selected_extended = CONFIG['model_type'] == 'trainable'
NUM_CLASSES = CONFIG['num_classes']
EPOCHS_PER_RUN = CONFIG['epochs']
RUN_TAG = model_name
CLASS_NAMES = [str(i) for i in range(NUM_CLASSES)]

# Output location (rename the subfolder freely; just update the string below).
try:
    _root = str(project_root)
except NameError:
    _root = os.getcwd()
REPORTS_DIR = os.path.join(_root, "reports", dataset_name, "quantum")
multi_run_dir = os.path.join(REPORTS_DIR, "multi_run", "raw")
os.makedirs(multi_run_dir, exist_ok=True)


def _build_run_model(seed):
    """Return (model, optimizer, model_name) freshly built for this run."""
    if _selected_extended:
        m = ExtentedQuantumModel(n_qubits=CONFIG['n_qubits'], n_layers=CONFIG['n_layers'],
                                 num_classes=CONFIG['num_classes'], patch_size=CONFIG['patch_size'],
                                 mode="2d", stride=None, use_parallel=True).to(device)
        name = f"extended_quantum_run{seed + 1}"
    else:
        rp = np.random.uniform(high=2 * np.pi, size=(CONFIG['n_layers'], CONFIG['n_qubits']))
        m = QuantumModel(n_qubits=CONFIG['n_qubits'], n_layers=CONFIG['n_layers'], rand_params=rp,
                         output_size=output_size, patch_size=CONFIG['patch_size'],
                         num_classes=CONFIG['num_classes']).to(device)
        with torch.no_grad():
            _ = m(torch.randn(1, CONFIG['input_channels'], input_h, input_w).to(device))  # lazily build FC layer
        name = f"quantum_run{seed + 1}"
    opt = optim.Adam(m.parameters(), lr=CONFIG['learning_rate'])
    return m, opt, name

def _get_predictions(model, dataloader, device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            y_pred.extend(outputs.argmax(dim=1).cpu().numpy().tolist())
            y_true.extend(np.asarray(labels).squeeze().reshape(-1).tolist())
    return np.array(y_true), np.array(y_pred)


def _plot_confusion_matrix(cm, save_path, title, class_names=None, normalize=False):
    cm = np.asarray(cm, dtype=float)
    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        disp = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums != 0)
        cell_fmt = lambda v: f"{v:.1%}"
    else:
        disp = cm
        cell_fmt = lambda v: f"{int(round(v))}"
    n = cm.shape[0]
    names = class_names if class_names is not None else [str(i) for i in range(n)]
    fig, ax = plt.subplots(figsize=(1.4 * n + 2.5, 1.4 * n + 2.5))
    im = ax.imshow(disp, cmap="Blues", vmin=0)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(names, rotation=45, ha="right"); ax.set_yticklabels(names)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    thresh = disp.max() / 2 if disp.max() > 0 else 0.5
    for i in range(n):
        for j in range(n):
            ax.text(j, i, cell_fmt(disp[i, j]), ha="center", va="center",
                    color="white" if disp[i, j] > thresh else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


run_results = []
run_confusion_matrices = []

for run_idx in range(N_RUNS):
    seed = run_idx
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    print(f"\n{'=' * 80}")
    print(f"RUN {run_idx + 1}/{N_RUNS} (seed={seed}) - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'=' * 80}\n")

    run_model, run_optimizer, run_model_name = _build_run_model(seed)
    run_criterion = nn.CrossEntropyLoss()

    best_path = os.path.join(multi_run_dir, f"best_{run_model_name}.pth")
    if os.path.isfile(best_path):
        os.remove(best_path)

    training_loop(
        model=run_model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=run_criterion,
        optimizer=run_optimizer,
        device=device,
        num_epochs=EPOCHS_PER_RUN,
        save_dir=multi_run_dir,
        model_name=run_model_name,
        extract_features=False,
        save_best_only=True,
        save_all_epochs=False,
    )

    if os.path.isfile(best_path):
        run_model.load_state_dict(torch.load(best_path, map_location=device))
    else:
        print(f"⚠ Warning: run {run_idx + 1} never beat the initial val_acc baseline — "
              f"testing with final-epoch weights instead of a best checkpoint.")
    run_model.eval()

    test_loss, test_acc, test_prec, test_rec, test_f1, test_auc = test_model(
        model=run_model,
        dataloader=test_loader,
        criterion=run_criterion,
        device=device,
        hyperparams={**CONFIG, "model": run_model_name},
        output_file=os.path.join(multi_run_dir, f"test_log_run{run_idx + 1}.txt"),
        extract_features=False,
        compute_detailed_metrics=True,
    )

    run_labels, run_preds = _get_predictions(run_model, test_loader, device)
    run_cm = confusion_matrix(run_labels, run_preds, labels=list(range(NUM_CLASSES)))
    run_confusion_matrices.append(run_cm)
    _plot_confusion_matrix(
        run_cm,
        save_path=os.path.join(multi_run_dir, f"confusion_matrix_run{run_idx + 1}.png"),
        title=f"Confusion Matrix - Run {run_idx + 1}",
        class_names=CLASS_NAMES,
    )

    run_record = {
        "run": run_idx + 1,
        "loss": test_loss,
        "accuracy": test_acc,
        "precision": test_prec,
        "recall": test_rec,
        "f1": test_f1,
        "auc": test_auc,
    }
    _gamma = getattr(run_model, "gamma", None)
    if _gamma is not None:
        run_record["gamma"] = float(_gamma.item()) if hasattr(_gamma, "item") else float(_gamma)
    run_results.append(run_record)

    print(f"\n✓ Run {run_idx + 1} test accuracy: {test_acc:.4f}")

print(f"\n{'=' * 80}")
print(f"All {N_RUNS} runs complete")
print(f"{'=' * 80}")

metrics_keys = ["loss", "accuracy", "precision", "recall", "f1", "auc"]
if all("gamma" in r for r in run_results):
    metrics_keys.append("gamma")

stats = {}
for key in metrics_keys:
    values = np.array([r[key] for r in run_results], dtype=float)
    n = len(values)
    mean = float(np.mean(values))
    std = float(np.std(values, ddof=1)) if n > 1 else 0.0
    if n > 1:
        sem = std / np.sqrt(n)
        t_crit = float(t_dist.ppf((1 + CONFIDENCE_LEVEL) / 2, df=n - 1))
        margin = t_crit * sem
    else:
        margin = 0.0
    stats[key] = {
        "mean": mean, "std": std,
        "min": float(np.min(values)), "max": float(np.max(values)),
        "ci_lower": mean - margin, "ci_upper": mean + margin,
    }

cm_sum = np.stack(run_confusion_matrices).sum(axis=0)
cm_sum_path = os.path.join(multi_run_dir, "confusion_matrix_sum.png")
_plot_confusion_matrix(cm_sum.astype(int), cm_sum_path,
                       f"Confusion Matrix - Sum over {N_RUNS} runs", CLASS_NAMES)
cm_mean_path = os.path.join(multi_run_dir, "confusion_matrix_mean_normalized.png")
_plot_confusion_matrix(cm_sum, cm_mean_path,
                       f"Confusion Matrix - Row-Normalized (mean over {N_RUNS} runs)",
                       CLASS_NAMES, normalize=True)

stats_path = os.path.join(REPORTS_DIR, f"multi_run_statistics_{RUN_TAG}.txt")
report_meta = {
    "Dataset": CONFIG['dataset'].upper(),
    "Model Variant": CONFIG['model_type'],
    "Model": ("ExtentedQuantumModel" if _selected_extended else "QuantumModel"),
    "Epochs per run": CONFIG['epochs'],
    "Batch Size": CONFIG['batch_size'],
    "Learning Rate": CONFIG['learning_rate'],
    "n_qubits": CONFIG['n_qubits'],
    "n_layers": CONFIG['n_layers'],
    "patch_size": CONFIG['patch_size'],
}
with open(stats_path, "w", encoding="utf-8") as f:
    f.write("=" * 80 + "\n")
    f.write(f"MULTI-RUN STATISTICS ({N_RUNS} runs)\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    for _k, _v in report_meta.items():
        f.write(f"{_k}: {_v}\n")
    f.write("\n")

    f.write("PER-RUN TEST RESULTS\n")
    f.write("-" * 80 + "\n")
    header = f"{'Run':<5}{'Loss':<10}{'Accuracy':<10}{'Precision':<12}{'Recall':<10}{'F1':<10}{'AUC':<10}"
    if "gamma" in metrics_keys:
        header += f"{'Gamma':<10}"
    f.write(header + "\n")
    for r in run_results:
        line = (f"{r['run']:<5}{r['loss']:<10.4f}{r['accuracy']:<10.4f}"
                f"{r['precision']:<12.4f}{r['recall']:<10.4f}{r['f1']:<10.4f}{r['auc']:<10.4f}")
        if "gamma" in metrics_keys:
            line += f"{r['gamma']:<10.4f}"
        f.write(line + "\n")
    f.write("\n")

    f.write(f"SUMMARY STATISTICS (mean +/- std, min, max, {int(CONFIDENCE_LEVEL * 100)}% CI [t-distribution])\n")
    f.write("-" * 80 + "\n")
    for key in metrics_keys:
        s = stats[key]
        f.write(f"{key.capitalize():<12}: {s['mean']:.4f} +/- {s['std']:.4f}"
                f"  (min: {s['min']:.4f}, max: {s['max']:.4f})"
                f"  {int(CONFIDENCE_LEVEL * 100)}% CI: [{s['ci_lower']:.4f}, {s['ci_upper']:.4f}]\n")
    f.write("\n")

    labels_for_header = CLASS_NAMES if CLASS_NAMES is not None else [str(i) for i in range(cm_sum.shape[0])]
    f.write(f"CONFUSION MATRIX - SUMMED OVER {N_RUNS} RUNS (rows = true, cols = predicted)\n")
    f.write("-" * 80 + "\n")
    f.write("            " + "".join(f"{lbl:>12}" for lbl in labels_for_header) + "\n")
    for lbl, row in zip(labels_for_header, cm_sum.astype(int)):
        f.write(f"{lbl:>12}" + "".join(f"{v:>12d}" for v in row) + "\n")
    f.write(f"\nSaved plot: {cm_sum_path}\n\n")

    f.write(f"CONFUSION MATRIX - ROW-NORMALIZED MEAN OVER {N_RUNS} RUNS\n")
    f.write("-" * 80 + "\n")
    cm_norm = cm_sum.astype(float) / cm_sum.sum(axis=1, keepdims=True)
    f.write("            " + "".join(f"{lbl:>12}" for lbl in labels_for_header) + "\n")
    for lbl, row in zip(labels_for_header, cm_norm):
        f.write(f"{lbl:>12}" + "".join(f"{v:>12.2%}" for v in row) + "\n")
    f.write(f"\nSaved plot: {cm_mean_path}\n")

    f.write("\n" + "=" * 80 + "\n")
    f.write("END OF REPORT\n")
    f.write("=" * 80 + "\n")

print(f"✓ Multi-run statistics saved to: {stats_path}")
print()
for key in metrics_keys:
    s = stats[key]
    print(f"  {key.capitalize():<12}: {s['mean']:.4f} +/- {s['std']:.4f}"
          f"  (min: {s['min']:.4f}, max: {s['max']:.4f})"
          f"  {int(CONFIDENCE_LEVEL * 100)}% CI: [{s['ci_lower']:.4f}, {s['ci_upper']:.4f}]")
print("\nSummed confusion matrix:")
print(cm_sum.astype(int))


RUN 1/5 (seed=0) - 2026-07-15 08:24:34

Epoch 1/20


KeyboardInterrupt: 